# Expense Forecasting — Model Personal Per User


## 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os
import pickle
from dataclasses import dataclass
from typing import Optional

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [2]:
SAVE_DIR   = 'saved_model'
LOOK_BACK  = 3
FEATURE_COLS = [
    'expense_log', 'txn_count', 'unique_categories',
    'lag_1', 'lag_2', 'lag_3',
    'rolling_mean_3', 'rolling_std_3',
    'month_sin', 'month_cos',
]

os.makedirs(SAVE_DIR, exist_ok=True)

## 2. Custom Components (Layer & Loss)

In [3]:
@tf.keras.utils.register_keras_serializable()
class LinearScaleLayer(layers.Layer):
    def __init__(self, **kwargs):
        super().__init__(**kwargs)

    def build(self, input_shape):
        self.scale = self.add_weight(name='scale', shape=(1,), initializer='ones',  trainable=True)
        self.bias  = self.add_weight(name='bias',  shape=(1,), initializer='zeros', trainable=True)

    def call(self, x):
        return x * self.scale + self.bias

    def get_config(self):
        return super().get_config()


@tf.keras.utils.register_keras_serializable()
class HuberLoss(keras.losses.Loss):
    def __init__(self, delta=1.0, **kwargs):
        super().__init__(**kwargs)
        self.delta  = delta
        self._huber = keras.losses.Huber(delta=delta)

    def call(self, y_true, y_pred):
        return self._huber(y_true, y_pred)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'delta': self.delta})
        return cfg

## 3. Load & Preprocessing Data

In [30]:
df = pd.read_csv('cleaned_data_v8.csv')
df['date']       = pd.to_datetime(df['date'])
df['year_month'] = df['date'].dt.to_period('M').dt.to_timestamp()

# Monthly aggregation
monthly = (
    df[df['type'] == 'expense']
    .groupby(['user_id', 'year_month'])['amount']
    .sum()
    .reset_index()
    .rename(columns={'amount': 'monthly_expense'})
)

txn_count  = df.groupby(['user_id','year_month']).size().reset_index(name='txn_count')
unique_cat = df.groupby(['user_id','year_month'])['category'].nunique().reset_index(name='unique_categories')

monthly = monthly.merge(txn_count,  on=['user_id','year_month'], how='left')
monthly = monthly.merge(unique_cat, on=['user_id','year_month'], how='left')
monthly = monthly.fillna(0)

print(f'Data loaded: {df.shape}')
print(f'Total users : {monthly["user_id"].nunique()}')

Data loaded: (111480, 9)
Total users : 20


## 4. Feature Engineering (per user)

In [5]:
def build_features(user_df: pd.DataFrame) -> pd.DataFrame:
    df = user_df.copy().sort_values('year_month').reset_index(drop=True)

    df['expense_log']    = np.log1p(df['monthly_expense'])
    df['lag_1']          = df['expense_log'].shift(1)
    df['lag_2']          = df['expense_log'].shift(2)
    df['lag_3']          = df['expense_log'].shift(3)
    df['rolling_mean_3'] = df['expense_log'].rolling(3).mean()
    df['rolling_std_3']  = df['expense_log'].rolling(3).std()
    df['month']          = pd.to_datetime(df['year_month']).dt.month
    df['month_sin']      = np.sin(2 * np.pi * df['month'] / 12)
    df['month_cos']      = np.cos(2 * np.pi * df['month'] / 12)
    df = df.fillna(0)
    return df

## 5. Fungsi Build Model GRU + Attention

In [6]:
def build_gru_model(n_features: int, look_back: int) -> keras.Model:
    seq_input = keras.Input(shape=(look_back, n_features), name='sequence_input')

    x = layers.GRU(64, return_sequences=True)(seq_input)
    x = layers.Dropout(0.2)(x)
    x = layers.GRU(32, return_sequences=True)(x)
    x = layers.Dropout(0.2)(x)

    # Multi-Head Attention
    attn = layers.MultiHeadAttention(num_heads=2, key_dim=16)(x, x)
    x    = layers.LayerNormalization()(x + attn)

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(32, activation='relu')(x)
    x = layers.Dropout(0.1)(x)
    x = layers.Dense(1)(x)

    output = LinearScaleLayer(name='linear_scale')(x)

    model = keras.Model(inputs=seq_input, outputs=output, name='GRU_Attention_PerUser')
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=0.0005),
        loss=HuberLoss(delta=1.0),
        metrics=['mae']
    )
    return model

## 6. Train Model Per User

Cell ini melatih satu model untuk setiap user dan menyimpannya ke `saved_model/<user_id>/`.

In [7]:
def train_user_model(user_id: str, user_monthly: pd.DataFrame, verbose: int = 0) -> dict:
    user_dir = os.path.join(SAVE_DIR, user_id)
    os.makedirs(user_dir, exist_ok=True)

    df = build_features(user_monthly)

    scaler = RobustScaler()
    df[FEATURE_COLS] = scaler.fit_transform(df[FEATURE_COLS])

    values  = df[FEATURE_COLS].values
    targets = df['expense_log'].values

    effective_look_back = min(LOOK_BACK, max(1, len(values) - 2))

    X, y = [], []
    for i in range(effective_look_back, len(values)):
        X.append(values[i-effective_look_back:i])
        y.append(targets[i])

    X = np.array(X)
    y = np.array(y)

    if len(X) == 0:
        return {'user_id': user_id, 'mae': None, 'rmse': None}

    use_val_split = len(X) >= 5
    split         = int(len(X) * 0.8) if use_val_split else len(X)
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    model = build_gru_model(
        n_features=len(FEATURE_COLS),
        look_back=effective_look_back
    )

    callbacks = [
        EarlyStopping(
            monitor='val_loss' if use_val_split else 'loss',
            patience=5,
            restore_best_weights=True
        ),
        ReduceLROnPlateau(
            monitor='val_loss' if use_val_split else 'loss',
            factor=0.5,
            patience=5,
            verbose=0
        ),
    ]

    model.fit(
        X_train,
        y_train,
        validation_split=0.2 if use_val_split else 0.0,
        epochs=100,
        batch_size=min(16, len(X_train)),
        callbacks=callbacks,
        verbose=verbose,
    )

    # Evaluasi
    if len(X_test) > 0:
        pred_log = model.predict(X_test, verbose=0).flatten()

        dummy = np.zeros((len(pred_log), len(FEATURE_COLS)))
        dummy[:, 0] = pred_log
        pred_real = np.expm1(scaler.inverse_transform(dummy)[:, 0])

        dummy2 = np.zeros((len(y_test), len(FEATURE_COLS)))
        dummy2[:, 0] = y_test
        true_real = np.expm1(scaler.inverse_transform(dummy2)[:, 0])

        mae  = mean_absolute_error(true_real, pred_real)
        rmse = np.sqrt(mean_squared_error(true_real, pred_real))
    else:
        mae, rmse = 0.0, 0.0

    model.save(os.path.join(user_dir, 'model.keras'))

    with open(os.path.join(user_dir, 'scaler.pkl'), 'wb') as f:
        pickle.dump(scaler, f)

    with open(os.path.join(user_dir, 'look_back.pkl'), 'wb') as f:
        pickle.dump(effective_look_back, f)

    return {
        'user_id': user_id,
        'mae': mae,
        'rmse': rmse
    }

In [8]:
# Train semua user
users = monthly['user_id'].unique()
results = []

print(f'Mulai training {len(users)} user...\n')

for i, uid in enumerate(users):
    user_data = monthly[monthly['user_id'] == uid].copy()

    result = train_user_model(
        uid,
        user_data,
        verbose=0
    )

    results.append(result)

    print(
        f'[{i+1:2d}/{len(users)}] '
        f'{uid} — MAE: Rp {result["mae"]:,.0f}'
    )

Mulai training 20 user...

[ 1/20] USER_001 — MAE: Rp 67,178
[ 2/20] USER_002 — MAE: Rp 65,682
[ 3/20] USER_003 — MAE: Rp 70,635
[ 4/20] USER_004 — MAE: Rp 66,620


[ 5/20] USER_005 — MAE: Rp 61,119


[ 6/20] USER_006 — MAE: Rp 95,195
[ 7/20] USER_007 — MAE: Rp 69,416
[ 8/20] USER_008 — MAE: Rp 58,746
[ 9/20] USER_009 — MAE: Rp 49,754
[10/20] USER_010 — MAE: Rp 78,614
[11/20] USER_011 — MAE: Rp 71,094
[12/20] USER_012 — MAE: Rp 85,332
[13/20] USER_013 — MAE: Rp 57,804
[14/20] USER_014 — MAE: Rp 74,809
[15/20] USER_015 — MAE: Rp 92,754
[16/20] USER_016 — MAE: Rp 93,499
[17/20] USER_017 — MAE: Rp 66,506
[18/20] USER_018 — MAE: Rp 80,664
[19/20] USER_019 — MAE: Rp 89,978
[20/20] USER_020 — MAE: Rp 79,470


## 7. Ringkasan Evaluasi Semua User

In [9]:
df_results = pd.DataFrame(results)
df_results = df_results.sort_values('mae')
df_results['mae']  = df_results['mae'].apply(lambda x: f'Rp {x:,.0f}')
df_results['rmse'] = df_results['rmse'].apply(lambda x: f'Rp {x:,.0f}')
df_results.columns = ['User ID', 'MAE', 'RMSE']
print(df_results.to_string(index=False))

 User ID       MAE       RMSE
USER_009 Rp 49,754  Rp 62,098
USER_013 Rp 57,804  Rp 81,735
USER_008 Rp 58,746  Rp 80,434
USER_005 Rp 61,119  Rp 69,269
USER_002 Rp 65,682  Rp 74,534
USER_017 Rp 66,506  Rp 77,272
USER_004 Rp 66,620  Rp 79,788
USER_001 Rp 67,178  Rp 82,950
USER_007 Rp 69,416  Rp 81,568
USER_003 Rp 70,635  Rp 86,135
USER_011 Rp 71,094  Rp 86,450
USER_014 Rp 74,809  Rp 97,153
USER_010 Rp 78,614  Rp 87,217
USER_020 Rp 79,470  Rp 93,091
USER_018 Rp 80,664  Rp 89,356
USER_012 Rp 85,332  Rp 91,764
USER_019 Rp 89,978 Rp 109,315
USER_015 Rp 92,754 Rp 106,920
USER_016 Rp 93,499 Rp 103,171
USER_006 Rp 95,195 Rp 111,392


## 8. Hybrid Predictor (Personal + Cold Start)

- User existing → load model personal
- User baru, data < 3 bulan → cold start
- User baru, data ≥ 3 bulan → **auto-train model personal baru**

In [10]:
@dataclass
class PredictionResult:
    user_id: str
    predicted_expense: float
    mode: str
    confidence: str
    months_of_data: int
    note: str

    def display(self):
        print('=' * 55)
        print(f'  User ID          : {self.user_id}')
        print(f'  Predicted Expense: Rp {self.predicted_expense:,.0f}')
        print(f'  Mode             : {self.mode}')
        print(f'  Confidence       : {self.confidence}')
        print(f'  Months of Data   : {self.months_of_data}')
        print(f'  Note             : {self.note}')
        print('=' * 55)


class PersonalModelPredictor:
    def __init__(self, user_id: str, save_dir: str):
        user_dir   = os.path.join(save_dir, user_id)
        self.model = keras.models.load_model(
            os.path.join(user_dir, 'model.keras'), compile=False
        )
        with open(os.path.join(user_dir, 'scaler.pkl'), 'rb') as f:
            self.scaler = pickle.load(f)

        look_back_path = os.path.join(user_dir, 'look_back.pkl')
        if os.path.exists(look_back_path):
            with open(look_back_path, 'rb') as f:
                self.look_back = pickle.load(f)
        else:
            self.look_back = LOOK_BACK

    def predict(self, user_monthly: pd.DataFrame) -> tuple:
        df     = build_features(user_monthly)
        values = df[FEATURE_COLS].values

        if len(values) < self.look_back:
            pad    = np.zeros((self.look_back - len(values), len(FEATURE_COLS)))
            values = np.vstack([pad, values])

        seq        = values[-self.look_back:]
        seq_scaled = self.scaler.transform(seq)
        X          = np.expand_dims(seq_scaled, axis=0)

        pred_scaled = self.model.predict(X, verbose=0).flatten()
        dummy       = np.zeros((1, len(FEATURE_COLS)))
        dummy[0, 0] = pred_scaled[0]
        pred_real   = float(np.expm1(self.scaler.inverse_transform(dummy)[0, 0]))

        months     = len(user_monthly)
        confidence = 'high' if months >= 12 else 'medium'
        return max(pred_real, 0.0), confidence


class ColdStartPredictor:

    def predict(
        self,
        onboarding_estimate: Optional[float],
        months_of_data: int,
        actual_expenses: Optional[list] = None,
    ) -> tuple:
        if months_of_data == 0:
            if onboarding_estimate and onboarding_estimate > 0:
                prediction = onboarding_estimate
                confidence = 'low'
            else:
                raise ValueError(
                    'User baru tanpa data aktual harus mengisi estimasi onboarding.'
                )
        else:
            actual_mean = np.mean(actual_expenses)

            if onboarding_estimate and onboarding_estimate > 0:
                w_actual  = 0.5 + (months_of_data * 0.15)
                w_onboard = 1.0 - w_actual
                prediction = w_actual * actual_mean + w_onboard * onboarding_estimate
            else:
                prediction = actual_mean

            confidence = 'medium' if months_of_data >= 2 else 'low'

        return max(prediction, 0.0), confidence


class HybridPredictor:
    TRAIN_THRESHOLD = 3

    def __init__(self, save_dir: str):
        self.save_dir = save_dir
        self.cold_start = ColdStartPredictor()
        self._cache     = {}

    def _has_personal_model(self, user_id: str) -> bool:
        return os.path.exists(os.path.join(self.save_dir, user_id, 'model.keras'))

    def _load_personal_model(self, user_id: str) -> PersonalModelPredictor:
        if user_id not in self._cache:
            self._cache[user_id] = PersonalModelPredictor(user_id, self.save_dir)
        return self._cache[user_id]

    def _auto_train(self, user_id: str, user_monthly: pd.DataFrame):
        print(f'  → Autotraining model personal untuk {user_id}...')
        train_user_model(user_id, user_monthly, verbose=0)
        print(f'  → Model {user_id} selesai di-training')

        if user_id in self._cache:
            del self._cache[user_id]

    def predict(
        self,
        user_id: str,
        user_history_df: Optional[pd.DataFrame] = None,
        onboarding_estimate: Optional[float] = None,
    ) -> PredictionResult:

        months = len(user_history_df) if user_history_df is not None else 0

        # Sudah punya model personal
        if self._has_personal_model(user_id):
            predictor          = self._load_personal_model(user_id)
            history            = user_history_df if user_history_df is not None else pd.DataFrame()
            predicted, conf    = predictor.predict(history)

            return PredictionResult(
                user_id=user_id,
                predicted_expense=round(predicted, 2),
                mode='personal_model',
                confidence=conf,
                months_of_data=months,
                note=f'Model personal {user_id} ({months} bulan data)',
            )

        # User baru, data cukup kemudian autotrain
        if months >= self.TRAIN_THRESHOLD:
            self._auto_train(user_id, user_history_df)

            predictor       = self._load_personal_model(user_id)
            predicted, conf = predictor.predict(user_history_df)

            return PredictionResult(
                user_id=user_id,
                predicted_expense=round(predicted, 2),
                mode='personal_model',
                confidence=conf,
                months_of_data=months,
                note=f'Model personal baru auto-trained ({months} bulan data)',
            )

        # User baru, data belum cukup lakukan cold start
        actual = (
            user_history_df['monthly_expense'].tolist()
            if months > 0 else None
        )

        predicted, conf = self.cold_start.predict(
            onboarding_estimate=onboarding_estimate,
            months_of_data=months,
            actual_expenses=actual,
        )

        src = []

        if onboarding_estimate:
            src.append('estimasi onboarding')

        if months > 0:
            src.append(f'{months} bulan data aktual')

        return PredictionResult(
            user_id=user_id,
            predicted_expense=round(predicted, 2),
            mode='cold_start',
            confidence=conf,
            months_of_data=months,
            note=f'Cold start: {", ".join(src)}',
        )

## 9. Init Predictor

In [11]:
predictor = HybridPredictor(save_dir=SAVE_DIR)

## 10. Contoh Prediksi
### 10a. User Existing (pakai model personal)

In [12]:
uid          = 'USER_001'
user_history = monthly[monthly['user_id'] == uid].copy()

result = predictor.predict(user_id=uid, user_history_df=user_history)
result.display()

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RobustScaler was fitted with feature names
  warnings.warn(


  User ID          : USER_001
  Predicted Expense: Rp 3,717,915
  Mode             : personal_model
  Confidence       : high
  Months of Data   : 60
  Note             : Model personal USER_001 (60 bulan data)


### 10b. User Baru — Baru Daftar (cold start)

In [13]:
result = predictor.predict(
    user_id='NEW_USER',
    onboarding_estimate=6_000_000,
)
result.display()

  User ID          : NEW_USER_001
  Predicted Expense: Rp 6,000,000
  Mode             : cold_start
  Confidence       : low
  Months of Data   : 0
  Note             : Cold start: estimasi onboarding


### 10c. User Baru — Sudah >= 3 Bulan (auto-train model personal)

In [27]:
# Simulasi user baru
history_new = pd.DataFrame({
    'year_month':        pd.date_range('2025-01-01', periods=3, freq='MS'),
    'monthly_expense':   [8_200_000, 8_100_000, 7_800_000],
    'txn_count':         [20, 24, 18],
    'unique_categories': [6, 7, 5],
})

result = predictor.predict(
    user_id='NEW_USER_TEST',
    user_history_df=history_new,
    onboarding_estimate=8_500_000,
)
result.display()

  → Auto-training model personal untuk NEW_USER_TEST...
  → Model NEW_USER_TEST selesai di-training


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RobustScaler was fitted with feature names
  warnings.warn(


  User ID          : NEW_USER_TEST
  Predicted Expense: Rp 8,114,754
  Mode             : personal_model
  Confidence       : medium
  Months of Data   : 3
  Note             : Model personal baru auto-trained (3 bulan data)
